# 🎨 CBAE Training on Google Colab (GPU)

This notebook runs the full CBAE multi-stage training pipeline on a Colab GPU.

**Before you start:**
1. Go to **Runtime → Change runtime type → GPU (T4)**
2. Push your latest code to GitHub
3. Upload your synthetic data `.h5` files (or generate them here)

## 1️⃣ Setup: Clone Repo & Install Dependencies

In [ ]:
# ── Verify GPU is available ─────────────────────────────────────────────
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

In [ ]:
# ── Clone your repo ─────────────────────────────────────────────────────
# Replace with your actual GitHub repo URL
!git clone https://github.com/AnjanaKvd/CBAE.git
%cd CBAE

In [ ]:
# ── Install dependencies ────────────────────────────────────────────────
!pip install -q h5py imageio matplotlib numpy Pillow torchdiffeq tqdm transformers
!pip install -q opencv-python-headless scikit-learn scipy

## 2️⃣ Upload Training Data

You have **two options** to get your `.h5` data files into Colab:

### Option A: Upload from Google Drive (Recommended for large files)

In [ ]:
# ── Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Copy your data from Drive to the project
# Adjust the source path to where you stored your .h5 files in Drive
import shutil, os

DRIVE_DATA_DIR = '/content/drive/MyDrive/CBAE_data'  # ← EDIT THIS

for stage_dir in ['clean', 'robustness', 'bridge']:
    src = os.path.join(DRIVE_DATA_DIR, stage_dir)
    dst = f'data/synthetic/{stage_dir}'
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        for f in os.listdir(src):
            if f.endswith('.h5'):
                shutil.copy2(os.path.join(src, f), dst)
        print(f"✅ Copied {len(os.listdir(dst))} files to {dst}")
    else:
        print(f"⚠️  {src} not found in Drive — skipping")

### Option B: Upload directly from your PC

In [ ]:
# ── Direct upload (for smaller datasets) ────────────────────────────────
# Uncomment and run this cell to upload .h5 files from your PC

# from google.colab import files
# import os
# os.makedirs('data/synthetic/clean', exist_ok=True)
# uploaded = files.upload()  # Select your .h5 files
# for fname in uploaded:
#     os.rename(fname, f'data/synthetic/clean/{fname}')
#     print(f'Moved {fname} → data/synthetic/clean/')

## 3️⃣ Verify Data

In [ ]:
import os

for stage in ['clean', 'robustness', 'bridge']:
    path = f'data/synthetic/{stage}'
    if os.path.exists(path):
        h5_files = [f for f in os.listdir(path) if f.endswith('.h5')]
        print(f"📂 {path}: {len(h5_files)} .h5 files")
    else:
        print(f"❌ {path}: directory not found")

## 4️⃣ Override Config for GPU Training

The default config uses CPU. We override it for Colab's GPU.

In [ ]:
# ── Override DEVICE to cuda ─────────────────────────────────────────────
import training.config as cfg

# Force GPU
cfg.DEVICE = 'cuda'

# You can also bump the batch size on GPU (T4 has 15GB VRAM)
# cfg.BATCH_SIZE = 4  # Try 4 — if OOM, keep at 2

print(f"Device:         {cfg.DEVICE}")
print(f"Batch size:     {cfg.BATCH_SIZE}")
print(f"Canvas size:    {cfg.CANVAS_SIZE}")
print(f"Sequence len:   {cfg.SEQUENCE_LENGTH}")
print(f"Clean epochs:   {cfg.N_EPOCHS_CLEAN}")
print(f"Robust epochs:  {cfg.N_EPOCHS_ROBUSTNESS}")
print(f"Bridge epochs:  {cfg.N_EPOCHS_BRIDGE}")

## 5️⃣ Stage 1 — Train on Clean Synthetic Data (50 epochs)

In [ ]:
# ── Stage 1: Clean synthetic ────────────────────────────────────────────
# Uses the trainer.py CLI — GPU-aware settings
!python -m training.trainer \
    --stage clean \
    --epochs 50 \
    --render_size 128 \
    --max_frames 192 \
    --save_every 10 \
    --checkpoint_dir checkpoints/stage1

## 6️⃣ Stage 2a — Fine-tune on Noisy Synthetic (30 epochs)

In [ ]:
# ── Stage 2a: Robustness fine-tuning ────────────────────────────────────
# Resume from best Stage 1 checkpoint

import glob
stage1_ckpts = sorted(glob.glob('checkpoints/stage1/model_epoch_*.pt'))
latest_ckpt = stage1_ckpts[-1] if stage1_ckpts else None
print(f"Resuming from: {latest_ckpt}")

if latest_ckpt:
    !python -m training.trainer \
        --stage robustness \
        --resume {latest_ckpt} \
        --epochs 30 \
        --render_size 128 \
        --max_frames 192 \
        --save_every 10 \
        --checkpoint_dir checkpoints/stage2
else:
    print("❌ No Stage 1 checkpoint found. Run Stage 1 first.")

## 7️⃣ Stage 2b — Bridge Fine-tuning (20 epochs)

In [ ]:
# ── Stage 2b: Bridge fine-tuning ────────────────────────────────────────
import glob
stage2_ckpts = sorted(glob.glob('checkpoints/stage2/model_epoch_*.pt'))
latest_ckpt = stage2_ckpts[-1] if stage2_ckpts else None
print(f"Resuming from: {latest_ckpt}")

if latest_ckpt:
    !python -m training.trainer \
        --stage bridge \
        --resume {latest_ckpt} \
        --epochs 20 \
        --render_size 128 \
        --max_frames 192 \
        --save_every 10 \
        --checkpoint_dir checkpoints/stage2b
else:
    print("❌ No Stage 2 checkpoint found. Run Stage 2a first.")

## 8️⃣ Stage 3 — Fine-tune on Real Data (Anita)

In [ ]:
# ── Stage 3: Mixed / Real data fine-tuning ──────────────────────────────
import glob
stage2b_ckpts = sorted(glob.glob('checkpoints/stage2b/model_epoch_*.pt'))
latest_ckpt = stage2b_ckpts[-1] if stage2b_ckpts else None
print(f"Resuming from: {latest_ckpt}")

if latest_ckpt:
    !python -m training.trainer \
        --stage mixed \
        --resume {latest_ckpt} \
        --epochs 30 \
        --render_size 128 \
        --max_frames 192 \
        --loss_velocity_weight 0.0 \
        --save_every 5 \
        --checkpoint_dir checkpoints/stage3
else:
    print("❌ No Stage 2b checkpoint found. Run Stage 2b first.")

## 9️⃣ Save Checkpoints to Google Drive

**Important**: Colab sessions disconnect after ~12 hours (free tier).  
Save checkpoints to Drive so you don't lose progress!

In [ ]:
# ── Save checkpoints to Google Drive ────────────────────────────────────
import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/CBAE_checkpoints'  # ← EDIT THIS
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

for stage_dir in ['stage1', 'stage2', 'stage2b', 'stage3']:
    src = f'checkpoints/{stage_dir}'
    dst = os.path.join(DRIVE_SAVE_DIR, stage_dir)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        n_files = len(os.listdir(dst))
        print(f"✅ Saved {n_files} files → {dst}")
    else:
        print(f"⏭️  {src} not found — skipping")

print("\n🎉 All checkpoints saved to Google Drive!")

## 📊 Monitor Training Progress

In [ ]:
# ── Plot training curves ────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

def plot_training_log(csv_path, title="Training Progress"):
    if not os.path.exists(csv_path):
        print(f"❌ {csv_path} not found")
        return
    
    df = pd.read_csv(csv_path)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    
    # Total loss
    axes[0].plot(df['epoch'], df['total'], 'b-', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Total Loss')
    axes[0].set_title('Total Loss')
    axes[0].grid(True, alpha=0.3)
    
    # Component losses
    for col in ['render', 'bcs', 'crs', 'temp', 'clip']:
        if col in df.columns:
            axes[1].plot(df['epoch'], df[col], label=col, linewidth=1.5)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Component Losses')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    
    # Epoch time
    axes[2].bar(df['epoch'], df['time_s'], color='orange', alpha=0.7)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Time (seconds)')
    axes[2].set_title('Epoch Duration')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot any available logs
import os
for stage in ['stage1', 'stage2', 'stage2b', 'stage3']:
    csv_path = f'checkpoints/{stage}/training_log.csv'
    if os.path.exists(csv_path):
        plot_training_log(csv_path, title=f'Stage: {stage}')

## ⏱️ Expected Training Times on Colab T4 GPU

| Stage | CPU (your PC) | Colab T4 GPU (expected) |
|-------|--------------|------------------------|
| Stage 1 (50 epochs) | ~16 hours | ~1-3 hours |
| Stage 2a (30 epochs) | ~10 hours | ~0.5-1.5 hours |
| Stage 2b (20 epochs) | ~6 hours | ~0.5-1 hour |
| Stage 3 (30 epochs) | ~10 hours | ~0.5-1.5 hours |
| **Total** | **~42 hours** | **~3-7 hours** |

> **Note**: Free Colab sessions have a ~12-hour limit. Save checkpoints to Google Drive regularly!